# MCQ Image Solver — Qwen2.5-VL-7B-Instruct

**Hardware**: 2× T4 GPU (30 GB VRAM total) · 30 GB RAM  
**Model**: `Qwen/Qwen2.5-VL-7B-Instruct` (~15 GB bfloat16)  
**Strategy**: Chain-of-thought visual reasoning → extract final answer letter

> Supports text, math formulas (LaTeX), science diagrams, tables — anything an MCQ can contain.

## Cell 1 — Install dependencies

In [1]:
# %%capture
!pip install transformers==4.51.3 accelerate==0.34.2 qwen-vl-utils einops
!pip install pillow requests

## Cell 2 — GPU check & environment

In [2]:
import torch

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")
print(f"GPUs     : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}  : {props.name}  {props.total_memory / 1e9:.1f} GB")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16   # T4 supports bf16 via emulation; falls back to fp16 gracefully
print(f"\nUsing device : {DEVICE} | dtype : {DTYPE}")
# path: /home/souradeep/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-7B-Instruct/snapshots/cc594898137f460bfe9f0759e9844b3ce807cfb5
# import torch


# print(f"PyTorch  : {torch.__version__}")
# print(f"CUDA     : {torch.version.cuda}")
# print(f"GPUs     : {torch.cuda.device_count()}")
# for i in range(torch.cuda.device_count()):
#     props = torch.cuda.get_device_properties(i)
#     print(f"  GPU {i}  : {props.name}  {props.total_memory / 1e9:.1f} GB")

# # ── GPU Selection ──────────────────────────────────────────────────────────────
# # Option 1: Single GPU by index
# GPU_IDS = [0,1,3,6]           # Use GPU 0 only

# # Option 2: Multiple specific GPUs (for DataParallel / DDP)
# # GPU_IDS = [0, 1]      # Use GPU 0 and GPU 1

# # Option 3: All available GPUs
# # GPU_IDS = list(range(torch.cuda.device_count()))

# # Option 4: CPU fallback (ignore GPUs)
# # GPU_IDS = []
# # ──────────────────────────────────────────────────────────────────────────────

# def setup_device(gpu_ids: list[int]):
#     if not gpu_ids or not torch.cuda.is_available():
#         return torch.device("cpu"), []

#     available = torch.cuda.device_count()
#     invalid = [g for g in gpu_ids if g >= available]
#     if invalid:
#         raise ValueError(f"Invalid GPU IDs {invalid}. Available GPUs: 0–{available - 1}")

#     # Primary device is always the first in the list
#     device = torch.device(f"cuda:{gpu_ids[0]}")

#     # Restrict visible GPUs via environment (affects libraries like NCCL)
#     import os
#     os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(map(str, gpu_ids))

#     return device, gpu_ids


# DEVICE, ACTIVE_GPUS = setup_device(GPU_IDS)
# DTYPE = torch.bfloat16

# print(f"\nUsing device : {DEVICE}")
# print(f"Active GPUs  : {ACTIVE_GPUS if ACTIVE_GPUS else 'None (CPU mode)'}")
# print(f"dtype        : {DTYPE}")

# # Usage hint for multi-GPU with DataParallel:
# # model = torch.nn.DataParallel(model, device_ids=ACTIVE_GPUS)
# # model = model.to(DEVICE)

PyTorch  : 2.11.0+cu130
CUDA     : 13.0
GPUs     : 1
  GPU 0  : NVIDIA GB10  128.5 GB

Using device : cuda | dtype : torch.bfloat16


## Cell 3 — Load model + processor

`device_map="auto"` spreads the model across both T4s automatically via HuggingFace `accelerate`.

In [3]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

print("Loading model …")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto",          # balances across 2×T4
    # ── optional: quantise to 4-bit to save ~7 GB if VRAM is tight ──
    # load_in_4bit=True,
    # bnb_4bit_compute_dtype=torch.bfloat16,
    local_files_only=True
)
model.eval()

print("Loading processor …")
processor = AutoProcessor.from_pretrained(MODEL_ID)

print("✓ Model ready")
print(f"Total params : {sum(p.numel() for p in model.parameters()) / 1e9:.2f} B")

/home/souradeep/miniconda3/envs/GP/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model …


Loading checkpoint shards: 100%|██████████| 5/5 [02:57<00:00, 35.51s/it]


Loading processor …


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


✓ Model ready
Total params : 8.29 B


## Cell 4 — Core solver function

In [23]:
import re
from pathlib import Path
from PIL import Image

# ─────────────────────────────────────────────
#  Prompt engineering
# ─────────────────────────────────────────────
SYSTEM_PROMPT = """You are an expert exam solver specialising in deep learning, machine learning, 
mathematics, physics, chemistry, biology, and logical reasoning — with particular depth in:
- Neural network architectures (CNNs, RNNs, Transformers, GANs, VAEs)
- Training techniques (backpropagation, optimizers, regularization, batch norm)
- Loss functions, metrics, and evaluation strategies
- Supervised, unsupervised, and reinforcement learning
- Statistics, probability, and linear algebra as applied to ML

STRICT OUTPUT FORMAT:
- Options are numbered 1, 2, 3, 4 (NOT letters A/B/C/D)
- Your LAST line MUST be exactly one of:
    ANSWER: 1   or   ANSWER: 2   or   ANSWER: 3   or   ANSWER: 4   or   ANSWER: 5
- Output ANSWER: 5 if you are not confident — skipping is safer than guessing wrong
- Do NOT output any other value — it counts as hallucination and costs −1 point
- Do NOT write anything after the ANSWER line

DECISION RULE:
- Answer (1/2/3/4) only if confidence ≥ 50%
- If confidence is below 50% after elimination, output ANSWER: 5"""

USER_PROMPT = (
    "Solve the MCQ in this image using the following steps:\n\n"
    "Step 1 — Question type: What kind of problem is this?\n"
    "         (calculation / conceptual / code / diagram / other)\n\n"
    "Step 2 — List options: Write out all options exactly as numbered (1/2/3/4).\n\n"
    "Step 3 — Core principle: State the formula, theorem, or definition needed.\n"
    "         If making any assumption, state it explicitly here.\n"
    "         Use the SAME formula for EVERY layer/step — never switch to a shortcut.\n\n"
    "Step 4 — Evaluate each option: Apply the formula numerically for each layer.\n"
    "         Show: formula → substitution → result, for every single step.\n"
    "         Then accept or eliminate each option with a specific reason.\n\n"
    "Step 5 — Verify: Cross-check your final result against ALL options.\n"
    "         If your result doesn't match any option exactly, recompute from Step 3.\n\n"
    "Step 6 — Confidence check: Rate your confidence (0–100%).\n"
    "         ≥ 50% → answer with 1/2/3/4\n"
    "         < 50% → output 5 (skip, no penalty)\n\n"
    "Final line must be: ANSWER: <1 or 2 or 3 or 4 or 5>"
)

from PIL import Image, ImageEnhance, ImageFilter

def preprocess_image(image_input, target_min_side: int = 1024) -> Image.Image:
    """
    Upscale small images, boost contrast, and sharpen before passing to the model.
    Qwen2.5-VL benefits from clear, high-contrast input.
    """
    if isinstance(image_input, (str, Path)):
        img = Image.open(image_input).convert("RGB")
    elif isinstance(image_input, Image.Image):
        img = image_input.convert("RGB")
    else:
        raise TypeError(f"Unsupported type: {type(image_input)}")

    # Upscale if too small — model resolution is key for text/formula legibility
    # w, h = img.size
    # min_side = min(w, h)
    # if min_side < target_min_side:
    #     scale = target_min_side / min_side
    #     img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    img = img.resize((448, 448), Image.LANCZOS)

    # Boost contrast slightly — helps with faint printed text
    img = ImageEnhance.Contrast(img).enhance(1.3)

    # Sharpen — helps with slightly blurry scans
    img = img.filter(ImageFilter.SHARPEN)

    return img

def build_messages(image_input):
    img = preprocess_image(image_input)   # ← add this line
    img_payload = {"type": "image", "image": img}
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [img_payload, {"type": "text", "text": USER_PROMPT}]},
    ]


def extract_answer(text: str) -> str:
    """
    Multi-stage parser — handles every common model output format.
    Priority order: canonical tag > last-line letter > bolded letter > first letter.
    """
    VALID = set("12345")

    # Stage 1: Canonical ANSWER: X tag (case-insensitive, flexible spacing)
    match = re.search(r"(?:ANSWER|ANS|Answer)\s*[:\-]\s*\(?([1-5])\)?", text)
    if match:
        return match.group(1).upper()

    # Stage 2: "The answer is X" / "Option X is correct" / "Therefore X"
    match = re.search(
        r"(?:the\s+(?:correct\s+)?answer\s+is|therefore[,\s]+(?:the\s+answer\s+is)?|option)\s*[:\-]?\s*\(?([1-5])\)?",
        text, re.IGNORECASE
    )
    if match:
        return match.group(1).upper()

    # Stage 3: Last non-empty line that is just a letter (with optional brackets/period)
    lines = [l.strip() for l in text.strip().splitlines() if l.strip()]
    for line in reversed(lines):
        m = re.match(r"^\(?([1-5])\)?[.):]?\s*$", line)
        if m and m.group(1).upper() in VALID:
            return m.group(1).upper()

    # Stage 4: Last occurrence of a standalone letter in the entire output
    matches = re.findall(r"\b([1-5])\b", text)
    if matches:
        return matches[-1].upper()

    return "UNKNOWN"


@torch.inference_mode()
def solve_mcq(image_input, max_new_tokens: int = 1024,
              temperature: float = 0.05, verbose: bool = True) -> dict:

    messages = build_messages(image_input)

    # Prepare text input
    text_input = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # Prepare image input
    image_inputs, video_inputs = process_vision_info(messages)

    # Tokenize everything
    inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    # Generate
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=(temperature > 0.0),
        repetition_penalty=1.05,
        top_p=0.9,
        top_k=20,
        pad_token_id=processor.tokenizer.eos_token_id,
    )

    # Decode only the newly generated tokens
    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    raw_output = processor.batch_decode(
        generated, skip_special_tokens=True, clean_up_tokenization_spaces=True
    )[0]

    answer = extract_answer(raw_output)

    if verbose:
        sep = "─" * 60
        print(sep)
        print("REASONING TRACE")
        print(sep)
        print(raw_output)
        print(sep)
        print(f"\n✅  FINAL ANSWER : {answer}\n")

    return {
        "answer": answer,
        # "reasoning": raw_output,
        # "raw_output": raw_output,
    }

print("✓ Solver function ready")

✓ Solver function ready


## Cell 5 — (Optional) Self-consistency voting

Run the model **N times** at higher temperature and pick the majority answer.  
Boosts accuracy on hard questions at the cost of N× inference time.

In [24]:
from collections import Counter

def solve_mcq_voting(image_input, n_votes: int = 5,
                     temperature: float = 0.4, verbose: bool = True) -> dict:
    votes, reasonings = [], []

    for i in range(n_votes):
        result = solve_mcq(image_input, temperature=temperature, verbose=False)
        votes.append(result["answer"])
        reasonings.append(result["reasoning"])
        if verbose:
            print(f"Vote {i+1}/{n_votes} → {result['answer']}")

    tally = Counter(votes)

    # Detect tie — run a tiebreaker at temperature=0 (greedy)
    top_count = tally.most_common(1)[0][1]
    top_answers = [a for a, c in tally.items() if c == top_count and a != "UNKNOWN"]

    if len(top_answers) > 1:
        if verbose:
            print(f"⚠️  Tie between {top_answers} — running greedy tiebreaker")
        tiebreaker = solve_mcq(image_input, temperature=0.0, verbose=False)
        winner = tiebreaker["answer"]
    else:
        # Exclude UNKNOWN from winning unless it's all we have
        non_unknown = [(a, c) for a, c in tally.most_common() if a != "UNKNOWN"]
        winner = non_unknown[0][0] if non_unknown else "UNKNOWN"

    best_reasoning = reasonings[votes.index(winner)] if winner in votes else reasonings[0]

    if verbose:
        print(f"\n📊 Vote tally : {dict(tally)}")
        print(f"✅  FINAL ANSWER (majority) : {winner}")

    return {"answer": winner}

print("✓ Voting solver ready")

✓ Voting solver ready


## Cell 6 — Batch solver (CSV pipeline)

Reads a CSV with columns `id, image_path` and writes predictions to `submissions.csv`.

In [25]:
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

def run_batch(
    csv_path: str,
    image_col: str = "image_path",
    id_col: str = "id",
    output_path: str = "submissions.csv",
    use_voting: bool = False,
    n_votes: int = 3,
) -> pd.DataFrame:
    """
    Process every row in a CSV and produce a submission file.

    Parameters
    ----------
    csv_path    : path to input CSV (columns: id, image_path)
    image_col   : column containing image file paths
    id_col      : column containing question IDs
    output_path : where to save submission CSV
    use_voting  : enable self-consistency voting (slower, more accurate)
    n_votes     : votes per question when use_voting=True
    """
    df = pd.read_csv(csv_path)
    results = []

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Solving MCQs"):
        qid   = row[id_col]
        img_p = row[image_col]
        try:
            if use_voting:
                out = solve_mcq_voting(img_p, n_votes=n_votes, verbose=False)
            else:
                out = solve_mcq(img_p, verbose=False)

            # ── Auto-retry with voting if we got UNKNOWN ──────────────────
            if out["answer"] == "UNKNOWN":
                print(f"[RETRY] id={qid} — UNKNOWN, switching to voting")
                out = solve_mcq_voting(img_p, n_votes=3, temperature=0.3, verbose=False)

            results.append({"id": qid, "answer": out["answer"]})
        except Exception as e:
            print(f"[ERROR] id={qid} — {e}")
            results.append({"id": qid, "answer": "UNKNOWN"})

print("✓ Batch solver ready")

✓ Batch solver ready


## Cell 7 — Single image inference demo

Replace the path (or URL) with your MCQ image.

In [7]:
#! wget https://i.ytimg.com/vi/wv7EV72K9kI/maxresdefault.jpg
#! wget https://scontent-bom5-2.xx.fbcdn.net/v/t39.30808-6/612308336_1283371180480424_2400012865101136602_n.jpg?stp=dst-jpg_p526x296_tt6&_nc_cat=100&ccb=1-7&_nc_sid=7b2446&_nc_ohc=osuNyAtZL1QQ7kNvwEyVOGv&_nc_oc=AdrzM-bG1KylLSAumL8bnRZh9966oBAPVPP0Ro1G1sHn1PYfSMIhctS_tBphHENRgmY&_nc_zt=23&_nc_ht=scontent-bom5-2.xx&_nc_gid=Rk-CW8vhIPpJZiNmvr0BCg&_nc_ss=7a389&oh=00_Af2e3tZtIsS9UHM4BVGM4lyNVUy6_2QyAplEdzKNrwx4Qg&oe=69DB4FBF

In [ ]:
import urllib.request ,os, time
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

# ── Option A: load from disk ───────────────────────────────────────────────
# IMAGE_PATH = "/kaggle/input/your-dataset/question_001.png"

# ── Option B: download a sample MCQ image to test ─────────────────────────
# SAMPLE_URL = (
#     "https://upload.wikimedia.org/wikipedia/commons/thumb/"
#     "8/8e/SATMathSampleQ.png/640px-SATMathSampleQ.png"
# )
# SAMPLE_PATH = "/tmp/sample_mcq.png"
# urllib.request.urlretrieve(SAMPLE_URL, SAMPLE_PATH)
IMAGE_PATH = "image3.png"

# Show the question
test_dir="all_images"
results = []

start_time = time.time()

for x in tqdm(os.listdir(test_dir)):

    IMAGE_PATH = os.path.join(test_dir, x)

    try:
        img = Image.open(IMAGE_PATH).convert("RGB")

        # ---- Solve ----
        result = solve_mcq(IMAGE_PATH, verbose=False)

        # Store result
        results.append({
            "image_name": x,
            "prediction": result['answer']
        })

    except Exception as e:
        print(f"Error processing {x}: {e}")
        results.append({
            "image_name": x,
            "prediction": None
        })

end_time = time.time()
total_time = end_time - start_time

# ---- Save CSV ----
df = pd.DataFrame(results)
df.to_csv("submission.csv", index=False)

# ---- Print time ----
print(f"\nTotal Time: {total_time:.2f} seconds")
print(f"Avg Time per Image: {total_time / len(results):.2f} seconds")

100%|██████████| 15/15 [16:51<00:00, 67.45s/it]


Total Time: 1011.72 seconds
Avg Time per Image: 67.45 seconds


: 

In [28]:
print(result['answer'])

3


## Cell 8 — Self-consistency demo (harder questions)

In [13]:
# Use 5-vote majority for hard / ambiguous MCQs
result_voted = solve_mcq_voting(IMAGE_PATH, n_votes=5, temperature=0.1, verbose=True)

Vote 1/5 → 2
Vote 2/5 → 2
Vote 3/5 → 2


KeyboardInterrupt: 

## Cell 9 — Batch run (Kaggle competition)

Uncomment and adjust paths when running on a real dataset.

In [ ]:
# submission = run_batch(
#     csv_path="/kaggle/input/competition-data/test.csv",
#     image_col="image_path",
#     id_col="id",
#     output_path="/kaggle/working/submission.csv",
#     use_voting=True,    # set False for speed; True for accuracy
#     n_votes=5,
# )

## Cell 10 — Memory diagnostics

In [17]:
for i in range(torch.cuda.device_count()):
    alloc   = torch.cuda.memory_allocated(i)   / 1e9
    reserved = torch.cuda.memory_reserved(i)   / 1e9
    total   = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"GPU {i} — allocated: {alloc:.2f} GB | reserved: {reserved:.2f} GB | total: {total:.2f} GB")

GPU 0 — allocated: 16.59 GB | reserved: 21.15 GB | total: 128.52 GB
